# FINN Dataflow Compiler: Analysis of the Heatmap Model

This notebook implements the FINN builder flow for a trained ONNX model.
The objectives are the transformation of the neural network into a streaming dataflow architecture,
the verification of numerical integrity, and the estimation of resource consumption on the FPGA.

In [ ]:
import os
import shutil
import json
import numpy as np
import onnxruntime as ort
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
from finn.util.visualization import showInNetron
from qonnx.util.cleanup import cleanup as qonnx_cleanup

# --- 1. Path Configuration ---
nb_dir = os.getcwd()
model_name = "untrained_heatmap_model.onnx" # Adjust name if necessary
model_path = os.path.join(nb_dir, model_name)
output_dir = os.path.join(nb_dir, "output_heatmap_estimate")

# Verify input file existence
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

print(f"Initializing build process for: {model_path}")

## 2. Model Cleanup
The model is cleaned to ensure that all tensor dimensions (shapes) are inferred
and the graph structure is optimized for the compiler.

In [ ]:
clean_model_path = model_path.replace(".onnx", "_clean.onnx")
qonnx_cleanup(model_path, out_file=clean_model_path)
print(f"Cleaned model saved: {clean_model_path}")

# Visualization of the input model
showInNetron(clean_model_path)

## 3. Generation of Reference Data for Verification
Since the model contains trained weights, it is crucial to ensure that the hardware transformation
does not alter the mathematical behavior. We generate a random input vector
and calculate the expected result using ONNX Runtime (software emulation).

In [ ]:
# Create ONNX Runtime session
sess = ort.InferenceSession(clean_model_path)
input_name = sess.get_inputs()[0].name
input_shape = sess.get_inputs()[0].shape

# Generate random input (Float32)
# Note: Handle shape for batch size 1 explicitly if defined as string
if isinstance(input_shape[0], str): input_shape[0] = 1 
input_data = np.random.rand(*input_shape).astype(np.float32)

# Calculate reference output
expected_output = sess.run(None, {input_name: input_data})[0]

# Save as .npy files (FINN expects these specific filenames)
np.save("input.npy", input_data)
np.save("expected_output.npy", expected_output)
print("Reference data (input.npy, expected_output.npy) generated.")

## 4. Definition of the Build Configuration
Here, the parameters for hardware synthesis are defined.
The focus is on estimation (`ESTIMATE_REPORTS`), not yet on bitfile generation.

In [ ]:
# Clean up output directory
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

cfg_estimates = build.DataflowBuildConfig(
    output_dir          = output_dir,
    mvau_wwidth_max     = 80,             # Parallelization of Matrix-Vector Units
    target_fps          = 1000,           # Target throughput (Frames per Second)
    synth_clk_period_ns = 10.0,           # Target clock period (100 MHz)
    fpga_part           = "xc7z020clg400-1", # Target platform (here: Pynq-Z1)
    
    # Standard steps for estimation only
    steps               = build_cfg.estimate_only_dataflow_steps,
    
    # Enable verification for critical intermediate steps
    verify_steps=[
        build_cfg.VerificationStepType.QONNX_TO_FINN_PYTHON,
        build_cfg.VerificationStepType.TIDY_UP_PYTHON,
        build_cfg.VerificationStepType.STREAMLINED_PYTHON,
    ],
    
    # Desired output artifacts
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

In [ ]:
%%time
# Execution of the builder flow
build.build_dataflow_cfg(clean_model_path, cfg_estimates);

## 5. Analysis of Results
Visualization of the transformed hardware graph and output of estimated resources.

In [ ]:
# Visualization of the intermediate step "Convert to HW"
intermediate_model = os.path.join(output_dir, "intermediate_models", "step_convert_to_hw.onnx")
if os.path.exists(intermediate_model):
    print("Loading hardware abstraction graph...")
    showInNetron(intermediate_model)

# Reading the resource report
res_file = os.path.join(output_dir, "report", "estimate_layer_resources.json")
if os.path.exists(res_file):
    with open(res_file, 'r') as f:
        res_data = json.load(f)
    print("\n--- Estimated Resource Usage (Total) ---")
    print(json.dumps(res_data["total"], indent=2))
else:
    print("No resource estimate found.")

# Reading verification results
verif_dir = os.path.join(output_dir, "verification_output")
if os.path.exists(verif_dir):
    print("\n--- Verification Status ---")
    print("Checked steps:")
    # List .npy files containing _SUCCESS or _FAIL
    files = [f for f in os.listdir(verif_dir) if "SUCCESS" in f or "FAIL" in f]
    for f in files:
        print(f" - {f}")